Chapter 5 Bonus: Pretraining Qwen3 on Unlabeled Data

In [ ]:
# 中文注释：检查复现 Qwen3 版第5章预训练流程所需的第三方库版本
from importlib.metadata import version

# 中文注释：与原书 GPT-2 流程相比，这里额外需要 tokenizers 库，
# 因为 Qwen3 使用基于 Hugging Face tokenizers 的 BPE 分词器（tokenizer.json），
# 而不是原书 GPT-2 所用的 tiktoken
pkgs = [
    "matplotlib",
    "numpy",
    "torch",
    "tokenizers",       # to implement the Qwen3 tokenizer
       ]
# 中文注释：逐个打印上面列表中每个库的已安装版本号，便于排查环境问题
for p in pkgs:
    print(f"{p} version: {version(p)}")

5.1 Evaluating generative text models
No code


5.1.1 Using Qwen3 to generate text

In [ ]:
# ===================== 中文导读 =====================
# 本单元格把 Qwen3 的实现拼在一起，依次包含：
#   1) Qwen3 模型结构定义：SwiGLU 前馈网络 FeedForward、RMSNorm、
#      RoPE 旋转位置编码、分组查询注意力 GroupedQueryAttention、
#      TransformerBlock，以及整体的 Qwen3Model；
#   2) 用 Qwen3-0.6B 的官方超参数构造一个「权重随机初始化」的 Qwen3Model
#      （只借用 Qwen3 的模型架构，并不加载 Hugging Face 上预训练好的权重）；
#   3) 基于 Hugging Face tokenizers 库封装 Qwen3Tokenizer，并从 Hub
#      下载官方 Qwen3-0.6B-Base 的 tokenizer.json（只下载分词器文件，不下载模型权重）；
#   4) 复用第4章的 generate_text_simple 贪婪解码函数，对随机初始化的模型
#      做一次生成演示（此时模型尚未训练，输出预计是不连贯的乱码文本）。
# 后续单元格会用这里定义的 Qwen3Model + Qwen3Tokenizer，在 the-verdict.txt
# 这个小语料上从零训练（预训练），复现第5章的训练流程。
######################
### Qwen3 Code
######################

import torch
import torch.nn as nn


# 中文注释：SwiGLU 前馈网络（Qwen3/Llama 系列常用结构），比第3/4章 GPT
# 用的 GELU-MLP 多了一路门控分支：fc1 做门控(gate)投影，fc2 做上投影(up)，
# 用 silu(fc1(x)) 逐元素乘以 fc2(x) 后，再由 fc3 投影回 emb_dim
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # 中文注释：x: (batch, seq_len, emb_dim) -> x_fc1: (batch, seq_len, hidden_dim)
        x_fc1 = self.fc1(x)
        # 中文注释：x_fc2: (batch, seq_len, hidden_dim)，用作逐元素相乘的“上投影”分支
        x_fc2 = self.fc2(x)
        # 中文注释：SiLU(x_fc1) 作为门控，逐元素乘以 x_fc2 -> (batch, seq_len, hidden_dim)
        x = nn.functional.silu(x_fc1) * x_fc2
        # 中文注释：fc3 把 hidden_dim 投影回 emb_dim -> (batch, seq_len, emb_dim)
        return self.fc3(x)


# 中文注释：RMSNorm——均方根归一化，相比 LayerNorm 不做去均值，只按均方根缩放，
# 计算量更小；qwen3_compatible=True 时会先转 float32 计算方差/开方以保证数值稳定，
# 最后再转换回输入原本的 dtype（例如 bfloat16）
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        # 中文注释：variance 对最后一维（emb_dim 或 head_dim）求均方值，
        # 形状为 (..., 1)，keepdim=True 便于后面广播相乘
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        # 中文注释：rsqrt(variance+eps) 相当于 1/RMS，用它缩放 x 实现归一化
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)


# 中文注释：预计算 RoPE（旋转位置编码）用的 cos/sin 表；每个 head 内部
# 按 head_dim 分组的不同频率随位置旋转，使注意力天然编码相对位置信息，
# 并支持一定程度的长上下文外推；返回的 cos, sin 形状均为
# (context_length, head_dim)，使用时按实际 seq_len 切片
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 中文注释：将预计算好的 cos/sin 应用到 queries 或 keys 上，实现旋转位置编码
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)


# 中文注释：分组查询注意力 GQA——让多个 query head 共享同一组 key/value head，
# 大幅减少 K/V 的参数量以及推理时 KV Cache 的显存占用；
# num_heads 个 query head 被分成 num_kv_groups 组，每组 group_size=num_heads//num_kv_groups
# 个 query head 共享 1 个 kv head；qk_norm=True 时会对每个 head 的 q/k 做 RMSNorm，
# 这是 Qwen3 相比很多同类模型的一个特有设计，用于稳定训练
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin):
        # 中文注释：x 形状为 (b, num_tokens, d_in)
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        # 中文注释：queries reshape 为 (b, num_heads, num_tokens, head_dim)，
        # 把 head 维度提到前面，便于按 head 并行计算注意力
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        # 中文注释：keys/values reshape 为 (b, num_kv_groups, num_tokens, head_dim)，
        # 注意此时 kv 的 head 数量比 queries 少（GQA 的核心区别）
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        # 中文注释：用 repeat_interleave 把每个 kv head 复制 group_size 次，
        # 把 keys/values 的 head 数从 num_kv_groups 扩展到 num_heads，与 queries 对齐，
        # 之后就可以按普通多头注意力的方式计算
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        # 中文注释：attn_scores 形状为 (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)
        # 中文注释：mask 是因果（causal）掩码，下三角为可见、上三角为不可见，
        # 把未来位置的注意力分数置为 -inf，防止当前位置看到未来 token
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        # 中文注释：attn_weights @ values -> (b, num_heads, num_tokens, head_dim)，
        # 再 transpose + reshape 合并所有 head，得到 (b, num_tokens, num_heads*head_dim=d_out)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)


# 中文注释：Transformer 解码器块，采用 Pre-Norm 结构——先归一化再进注意力/前馈，
# 且每个子层都有残差连接（shortcut），思路与第4章 GPT 的 TransformerBlock 一致，
# 区别在于这里的注意力是 GQA、归一化用 RMSNorm，并且需要额外传入 RoPE 的 cos/sin
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x


# 中文注释：完整的 Qwen3 解码器模型——
# 词嵌入 -> N 层 TransformerBlock -> 最终 RMSNorm -> 输出线性层（得到词表 logits）
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Main model parameters
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # Uncomment the following code to tie weights
        # self.out_head.weight = self.tok_emb.weight
        # torch.nn.init.normal_(self.out_head.weight, mean=0.0, std=0.02)

        # Reusable utilities
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        # 中文注释：按配置里的 head_dim/rope_base/context_length 预计算好 RoPE 表，
        # 存成非持久 buffer（persistent=False），即不会被保存进 state_dict，
        # 因为这些值可以根据 config 重新计算出来，无需随权重一起保存/加载
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    def forward(self, in_idx):
        # Forward pass
        # 中文注释：in_idx 形状为 (batch, num_tokens)，是 token id 序列；
        # tok_embeds / x 形状为 (batch, num_tokens, emb_dim)
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        # 中文注释：因果掩码——上三角（不含对角线）为 True 表示要屏蔽（看不到未来），
        # 形状为 (num_tokens, num_tokens)，所有层共享同一个 mask
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)

        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        # 中文注释：logits 形状为 (batch, num_tokens, vocab_size)，
        # 即每个位置对词表中每个 token 的预测分数
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits
# 中文注释：用 Qwen3-0.6B 官方发布的超参数构造 QWEN3_CONFIG
# （词表大小、层数、维度等均与 Hugging Face 上的 Qwen/Qwen3-0.6B-Base 保持一致），
# 但下面只是用这份 config 随机初始化权重，并不会加载该仓库里预训练好的权重文件
#######################
### Initialize Qwen3
#######################

# 0.6B model
QWEN3_CONFIG = {
    "vocab_size": 151_936,           # Vocabulary size
    "context_length": 40_960,        # Context length that was used to train the model
    "emb_dim": 1024,                 # Embedding dimension
    "n_heads": 16,                   # Number of attention heads
    "n_layers": 28,                  # Number of layers
    "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
    "head_dim": 128,                 # Size of the heads in GQA
    "qk_norm": True,                 # Whether to normalize queries and keys in GQA
    "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
    "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
    "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
}

# 中文注释：额外定义 train_context_length=256，用于本notebook训练/推理时的
# 上下文窗口长度，它比 config 里 Qwen3 原生的 context_length=40_960 小很多，
# 目的是在小数据集、有限显存/内存条件下把训练开销控制在合理范围内
QWEN3_CONFIG["train_context_length"] = 256  # It's a small dataset, and we also want to keep memory usage reasonable

# 中文注释：固定随机种子，使模型权重的随机初始化可复现
torch.manual_seed(123)
# 中文注释：用随机初始化的权重构造 Qwen3 模型（此时尚未加载任何预训练权重）
model = Qwen3Model(QWEN3_CONFIG)
model.eval();
# 中文注释：用 Hugging Face tokenizers 库封装一个兼容 Qwen3 的分词器——
# 底层仍是标准 BPE（由 tokenizer.json 描述），但额外处理 Qwen 系列的
# 特殊 token（如 <|endoftext|>、<|im_start|>/<|im_end|>、<think>/</think> 等）
#######################
### Set up tokenizer
#######################

import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    # 中文注释：Qwen 系列模型定义的一批特殊 token，会被直接映射为固定 id，
    # 不参与常规的 BPE 子词切分
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    # 中文注释：用正则先把文本中的特殊 token 切分出来，其余普通文本片段
    # 再交给底层 BPE 分词器编码，避免特殊 token 被当作普通文本拆成子词
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        # 中文注释：加载本地 tokenizer.json（BPE 词表+合并规则）；
        # 这里用到的 Path 由本单元格后面的 `from pathlib import Path` 提供——
        # 虽然写在类定义之后，但函数体只在被真正调用时才执行，届时 Path 已经导入，
        # 因此不会报错
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        # 中文注释：默认用 <|endoftext|> 作为 pad/eos token
        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        # 中文注释：如果不是 Base（基座）模型，而是 Chat/Instruct 类模型，
        # 则用 <|im_end|> 作为句子结束符；本notebook使用的是
        # Qwen3-0.6B-Base，仍然使用 <|endoftext|>
        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        # 中文注释：若整段文本（去除首尾空白后）正好就是某个特殊 token，
        # 直接返回该 token 对应的 id，不再走 BPE 编码
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        # 中文注释：apply_chat_template=True 时，会把原始文本包装成 Qwen 的对话模板格式
        # （本notebook的预训练流程用的是 apply_chat_template=False，走纯文本续写）
        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        # 中文注释：按特殊 token 的位置切分文本——特殊 token 片段直接查表得到 id，
        # 普通文本片段用底层 BPE 分词器 (_tok) 编码
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        # 中文注释：skip_special_tokens=False，解码时保留特殊 token 对应的文本，
        # 便于调试时直接观察
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        # 中文注释：构造 Qwen 对话模板 <|im_start|>user ... <|im_end|>；
        # 若需要生成回复（add_generation_prompt=True），再追加 assistant 起始标记；
        # add_thinking 控制是否保留 Qwen3 的 <think>...</think> 推理段（本notebook未开启）
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s
# 中文注释：说明——Path/hf_hub_download 这两个 import 写在 Qwen3Tokenizer 类定义之后，
# 由于 Python 只在真正调用函数体时才解析名字，所以不影响下面代码的运行
from pathlib import Path
from huggingface_hub import hf_hub_download

# 中文注释：指定要使用的 Qwen3 官方模型仓库——这里只用到其中的 tokenizer.json，
# 不会下载/加载该仓库里的模型权重文件
repo_id = "Qwen/Qwen3-0.6B-Base"
tokenizer_file_path = "tokenizer.json"
local_dir = "."

# 中文注释：从 Hugging Face Hub 下载 tokenizer.json 到本地当前目录（local_dir="."）
hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

# 中文注释：实例化分词器；apply_chat_template=False 表示按纯文本（非对话模板）方式编码，
# 与第5章原始预训练流程保持一致（不引入对话/指令格式）
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=False,
    add_generation_prompt=False,
    add_thinking=False
)
# Same as chapter 4

# 中文注释：与第4章完全相同的贪婪解码（greedy decoding）生成函数，与具体模型架构无关，
# 只要传入的 model 实现了 forward(idx)->logits 接口即可直接复用
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


# 中文注释：字符串 -> (1, seq_len) 的 token id 张量（batch 维=1）
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

# 中文注释：(1, seq_len) 的 token id 张量 -> 字符串，去掉 batch 维后再 decode
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

# 中文注释：用尚未训练（权重随机初始化）的 Qwen3 模型做一次生成演示，
# 预期输出是不连贯的乱码文本——这里只是验证模型结构、分词器和生成函数能跑通，
# 真正的训练在后面的单元格进行
start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

5.1.2 Calculating the text generation loss: cross-entropy and perplexity
Similar to chapter 5


5.1.3 Calculating the training and validation set losses

In [ ]:
# 中文注释：下载/加载预训练语料——与第2章使用的 the-verdict.txt 完全相同的小说文本，
# 在本notebook中作为 Qwen3 架构“从零预训练”的训练语料
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
# 中文注释：打印文本开头片段，确认语料加载正确
# First 99 characters
print(text_data[:99])

In [ ]:
# 中文注释：打印文本结尾片段，确认语料加载正确
# Last 99 characters
print(text_data[-99:])

In [ ]:
# 中文注释：用 Qwen3Tokenizer 对整段语料编码，统计字符数与 token 数
# 注意：这里的 tokenizer 是前面从 Hugging Face 下载的 Qwen3 BPE 分词器，
# 其词表规模（151,936）远大于原书 GPT-2 的 tiktoken 词表（50,257），
# 因此同一段文本编码出来的 token 数量与原书结果会不同
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
# 中文注释：数据集/数据加载器定义，与第2章 GPTDatasetV1 完全一致，
# 采用滑动窗口构造 (input, target) 样本对，target 是 input 整体右移一位（预测下一个 token）
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        # 中文注释：这里的 tokenizer 参数实际传入的是 Qwen3Tokenizer 实例，
        # 只要具备 .encode() 接口即可复用同一套 Dataset 实现，与具体使用
        # GPT-2 还是 Qwen3 分词器无关（token_ids 是 Python list[int]）
        token_ids = tokenizer.encode(txt)

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        # 中文注释：max_length 即每个训练样本的序列长度
        # （下面会传入 QWEN3_CONFIG["train_context_length"]=256），
        # stride 控制滑动窗口步长；后面训练/验证 DataLoader 都用 stride==max_length，
        # 即样本之间不重叠切块
        for i in range(0, len(token_ids) - max_length, stride):
            # 中文注释：input_chunk 长度为 max_length；
            # target_chunk 是 input_chunk 整体右移一位，即标准的
            # next-token-prediction（预测下一个 token）构造方式
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Note that we have to change the function below because we previously hard-coded the
# GPT-2 tokenizer in the data loader
def create_dataloader_v1(txt, tokenizer, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    # tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = tokenizer

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


# Train/validation ratio
# 中文注释：按 90%/10% 划分训练集/验证集
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


# 中文注释：固定随机种子，使 DataLoader 内部 shuffle 的顺序可复现
torch.manual_seed(123)

# 中文注释：训练集 DataLoader —— max_length/stride 都用 train_context_length(256)，
# 且 stride==max_length 表示切块之间不重叠；drop_last=True 会丢弃
# 最后一个凑不满一整个 batch 的数据
train_loader = create_dataloader_v1(
    train_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=QWEN3_CONFIG["train_context_length"],
    stride=QWEN3_CONFIG["train_context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

# 中文注释：验证集 DataLoader —— shuffle=False 便于稳定评估，
# drop_last=False 保留全部数据（含不满一个 batch 的部分）
val_loader = create_dataloader_v1(
    val_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=QWEN3_CONFIG["train_context_length"],
    stride=QWEN3_CONFIG["train_context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 中文注释：打印各个 batch 的张量形状，验证 DataLoader 输出符合预期：
# x, y 形状均为 (batch_size, train_context_length) = (2, 256)
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
# 中文注释：统计训练/验证集里总的 token 数量，
# 用于估算数据规模、以及每个 epoch 大约会“看到”多少个 token
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

In [ ]:
# 中文注释：定义训练用的损失函数——标准的“预测下一个 token”交叉熵损失
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    # 中文注释：logits 形状为 (batch, seq_len, vocab_size)
    logits = model(input_batch)
    # 中文注释：flatten(0, 1) 把 (batch, seq_len, vocab_size) 展平成
    # (batch*seq_len, vocab_size)；target_batch.flatten() 展平成 (batch*seq_len,)，
    # 这样才能喂给 cross_entropy 计算逐 token 的分类损失
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


# 中文注释：对整个 DataLoader（或其中前 num_batches 个 batch）计算平均损失，
# 常用于训练过程中定期评估 train/val loss
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
# 中文注释：选择运行设备——优先 CUDA，其次 Apple Silicon 的 MPS，否则退回 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"Using {device} device.")


# 中文注释：训练前先计算一遍“未训练”（随机初始化权重）模型在训练/验证集上的
# 平均 loss，作为后续训练过程中对比的基线（baseline）
model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

5.2 Training an LLM

In [ ]:
# 中文注释：标准训练循环——与第5章主线内容完全一致的 train_model_simple，
# 这里的 model 换成了 Qwen3Model，calc_loss_batch / generate_text_simple 等
# 工具函数无需任何改动即可直接复用（说明训练循环本身与具体模型架构解耦）
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


# 中文注释：只用少量 batch（数量为 eval_iter）快速估算当前 train/val loss，
# 避免每次评估都要跑完整个 DataLoader，从而加快训练过程中的评估速度
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


# 中文注释：每个 epoch 结束后生成一段文本，直观查看模型在训练过程中
# 生成质量的变化情况
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    # 中文注释：这里用的是 config 中 Qwen3 原生的 context_length(=40_960)，
    # 而不是本notebook训练时实际使用的 train_context_length(=256)；
    # 由于 generate_text_simple 内部会用 idx[:, -context_size:] 做裁剪，
    # 而生成阶段序列长度远小于 40_960，实际上并不会真正触发裁剪，可以正常运行，
    # 但这与训练时的上下文长度并不完全一致，是需要留意的一处不一致（非报错风险项，不影响本notebook功能）
    context_size = model.cfg["context_length"]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()
# Note:
# Uncomment the following code to calculate the execution time
# import time
# start_time = time.time()

# 中文注释：重新固定随机种子，并重新构造一个全新的 Qwen3Model
# （丢弃前面演示用的模型实例，权重重新随机初始化），用于正式开始训练
torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)
model.to(device)
# 中文注释：AdamW 优化器，学习率与权重衰减沿用第5章的经验设置
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

# 中文注释：这里的训练轮数比第5章主线示例（通常10轮）更多，
# 因为 Qwen3 的词表（151,936）比 GPT-2（50,257）大很多，
# 初期需要更多训练步数才能把 loss 降下来
num_epochs = 40
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# 中文注释：与第5章相同的训练/验证 loss 可视化——
# 横轴同时显示 epoch 数和累计训练 token 数（共享同一个 y 轴）
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

5.3 Decoding strategies to control randomness

In [ ]:
# 中文注释：训练结束后，切到 CPU 上做推理演示（不依赖训练时用到的 GPU/MPS 设备）
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

# 中文注释：这里用回训练时的 train_context_length(256) 作为 context_size，
# 与训练阶段保持一致
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 中文注释：换一个新的起始文本再做一次生成演示，进一步观察训练效果
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Painting", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))